In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/xai-brain-tumor'

import os
if not os.path.exists('/content/xai-brain-tumor'):
    !git clone https://github.com/otaviotex/xai-brain-tumor
else:
    !git -C /content/xai-brain-tumor pull

import sys
sys.path.append('/content/xai-brain-tumor')

In [ ]:
import random, numpy as np, torch
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

from src.dataset import load_cache, make_loaders
cache = load_cache(f'{BASE}/data/cache_224.npz')
train_ds, val_ds, test_ds, train_dl, val_dl, test_dl = make_loaders(
    cache, f'{BASE}/data/splits', seed=SEED)
print('treino:', len(train_ds), '| validação:', len(val_ds), '| teste:', len(test_ds))

In [ ]:
import torch.nn as nn, pandas as pd
from torchvision.models import resnet18, ResNet18_Weights

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def build_model(estrategia):
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 3)
    if estrategia == 'A':   # congelada: só a camada final treina
        for n, p in m.named_parameters():
            p.requires_grad = n.startswith('fc.')
    return m.to(device)

for e in ['A', 'B']:
    m = build_model(e)
    print(e, 'parâmetros treináveis:', sum(p.numel() for p in m.parameters() if p.requires_grad))

# Etapa 7: pesos inversamente proporcionais à frequência no treino
ytr = pd.read_csv(f'{BASE}/data/splits/train.csv')['label'].values
cont = np.bincount(ytr, minlength=3)
w = len(ytr) / (3 * cont)
print('contagem no treino (meningioma, glioma, pituitary):', cont.tolist())
print('pesos de classe:', w.round(3).tolist())
criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(device))

In [ ]:
!git -C /content/xai-brain-tumor pull
import sys
sys.modules.pop('src.dataset', None)
sys.modules.pop('src', None)

In [ ]:
import os, json
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

print('device:', device)
MODELS_DIR = f'{BASE}/models'
MET_DIR = f'{BASE}/results/metrics'
os.makedirs(MODELS_DIR, exist_ok=True); os.makedirs(MET_DIR, exist_ok=True)
EPOCHS, PATIENCE = 30, 5
LR = {'A': 1e-3, 'B': 1e-4}

def train_one_epoch(model, dl, optimizer, estrategia):
    model.train(estrategia != 'A')   # em A a backbone fica em eval (BatchNorm fixa)
    loss_sum, correct, n = 0.0, 0, 0
    for x, y, _ in dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return loss_sum / n, correct / n

def evaluate(model, dl):
    model.eval()
    ys, ps, loss_sum, n = [], [], 0.0, 0
    with torch.no_grad():
        for x, y, _ in dl:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += criterion(out, y).item() * len(y); n += len(y)
            ys.append(y.cpu()); ps.append(out.argmax(1).cpu())
    y_true, y_pred = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return loss_sum / n, accuracy_score(y_true, y_pred), p, r, f1

def fit(estrategia):
    set_seed()
    _, _, _, train_dl, val_dl, _ = make_loaders(cache, f'{BASE}/data/splits', seed=SEED)
    model = build_model(estrategia)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=LR[estrategia])
    ckpt = f'{MODELS_DIR}/resnet18_{estrategia}_best.pt'
    best_f1, wait, rows = -1.0, 0, []
    for ep in range(1, EPOCHS + 1):
        tl, ta = train_one_epoch(model, train_dl, optimizer, estrategia)
        vl, va, vp, vr, vf = evaluate(model, val_dl)
        rows.append(dict(epoca=ep, train_loss=tl, val_loss=vl, train_acc=ta, val_acc=va,
                         val_precision_macro=vp, val_recall_macro=vr, val_macro_f1=vf))
        pd.DataFrame(rows).to_csv(f'{MET_DIR}/historico_{estrategia}.csv', index=False)
        if vf > best_f1:
            best_f1, wait = vf, 0
            torch.save({'model': model.state_dict(), 'epoca': ep, 'val_macro_f1': vf,
                        'estrategia': estrategia}, ckpt)
            marca = '  <- melhor'
        else:
            wait += 1; marca = ''
        print(f'[{estrategia}] época {ep:02d} | loss {tl:.4f}/{vl:.4f} | acc {ta:.3f}/{va:.3f} | val macro-F1 {vf:.4f}{marca}')
        if wait >= PATIENCE:
            print(f'[{estrategia}] early stopping na época {ep}'); break
    return pd.DataFrame(rows), best_f1

In [ ]:
hist_A, f1_A = fit('A')
hist_B, f1_B = fit('B')

In [ ]:
def plot_curves(hist, e):
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    axs[0].plot(hist.epoca, hist.train_loss, label='treino'); axs[0].plot(hist.epoca, hist.val_loss, label='validação')
    axs[0].set_title(f'Loss (estratégia {e})'); axs[0].set_xlabel('época'); axs[0].legend()
    axs[1].plot(hist.epoca, hist.train_acc, label='acc treino'); axs[1].plot(hist.epoca, hist.val_acc, label='acc validação')
    axs[1].plot(hist.epoca, hist.val_macro_f1, label='Macro F1 validação')
    axs[1].set_title(f'Métricas (estratégia {e})'); axs[1].set_xlabel('época'); axs[1].legend()
    plt.tight_layout(); plt.savefig(f'{BASE}/results/curvas_{e}.png', dpi=150); plt.show()

plot_curves(hist_A, 'A'); plot_curves(hist_B, 'B')

print('Melhor Macro F1 de validação: A =', round(f1_A, 4), '| B =', round(f1_B, 4))
escolhida = 'A' if f1_A >= f1_B else 'B'
print('Estratégia escolhida (pela validação):', escolhida)

cfg_treino = dict(
    pesos_iniciais='ResNet18 IMAGENET1K_V1', camada_final='Linear(512, 3)',
    estrategia_A='backbone congelada (BatchNorm em eval); treina só a fc',
    estrategia_B='todas as camadas treináveis', optimizer='Adam', lr=LR,
    loss='CrossEntropyLoss com pesos de classe n/(3*n_classe)', pesos_classe=w.round(4).tolist(),
    batch=32, epocas_max=EPOCHS, early_stopping_paciencia=PATIENCE,
    criterio_selecao='maior Macro F1 de validação', seed=SEED,
    melhor_val_macro_f1={'A': float(f1_A), 'B': float(f1_B)}, estrategia_escolhida=escolhida,
)
json.dump(cfg_treino, open(f'{MET_DIR}/config_treino.json', 'w'), indent=2, ensure_ascii=False)
print(json.dumps(cfg_treino, indent=2, ensure_ascii=False))

Se a sessão reiniciar, rode a célula de retomada. Não use Executar tudo, porque retreina os modelos.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/xai-brain-tumor'

import os, sys, json, random
if not os.path.exists('/content/xai-brain-tumor'):
    !git clone https://github.com/otaviotex/xai-brain-tumor.git
else:
    !git -C /content/xai-brain-tumor pull
sys.path.append('/content/xai-brain-tumor')

import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from torchvision.models import resnet18, ResNet18_Weights

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
MET_DIR = f'{BASE}/results/metrics'
os.makedirs(MET_DIR, exist_ok=True)

def build_model(estrategia):
    m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 3)
    if estrategia == 'A':
        for n, p in m.named_parameters():
            p.requires_grad = n.startswith('fc.')
    return m.to(device)

from src.dataset import load_cache, make_loaders
cache = load_cache(f'{BASE}/data/cache_224.npz')
train_ds, val_ds, test_ds, train_dl, val_dl, test_dl = make_loaders(
    cache, f'{BASE}/data/splits', seed=SEED)
print('treino:', len(train_ds), '| validação:', len(val_ds), '| teste:', len(test_ds))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 21 (delta 9), reused 18 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (21/21), 36.78 KiB | 649.00 KiB/s, done.
From https://github.com/otaviotex/xai-brain-tumor
   4db39a8..82fd28a  main       -> origin/main
Updating 4db39a8..82fd28a
Fast-forward
 results/{ => dataset}/batch_treino.png           | Bin
 results/{ => dataset}/dataset_analise.png        | Bin
 results/{ => dataset}/dataset_exemplos.png       | Bin
 results/{ => estrategia_A}/MetricasCurvaA.png    | Bin
 results/{ => estrategia_B}/MetricasCurvaB.png    | Bin
 "results/teste/matriz_confus\303\243o_teste.png" | Bin 0 -> 28846 bytes
 results/teste/metricas_teste.json                |   9 +
 results/teste/predicoes_teste.csv                | 4

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import torch.nn.functional as F

nomes = {0: 'meningioma', 1: 'glioma', 2: 'pituitary'}
ckpt = torch.load(f'{BASE}/models/resnet18_B_best.pt', map_location=device)
print('checkpoint: estratégia', ckpt['estrategia'], '| época', ckpt['epoca'],
      '| val macro-F1', round(ckpt['val_macro_f1'], 4))
model = build_model('B'); model.load_state_dict(ckpt['model']); model.eval()

ids, ys, ps, probs = [], [], [], []
with torch.no_grad():
    for x, y, m, i in test_dl:
        p = F.softmax(model(x.to(device)), dim=1).cpu()
        probs.append(p); ys.append(y); ps.append(p.argmax(1)); ids += i.tolist()
probs = torch.cat(probs).numpy()
y_true = torch.cat(ys).numpy(); y_pred = torch.cat(ps).numpy()

nm = list(nomes.values())
print(classification_report(y_true, y_pred, target_names=nm, digits=4))
rel = classification_report(y_true, y_pred, target_names=nm, digits=4, output_dict=True)
json.dump(rel, open(f'{MET_DIR}/metricas_teste.json', 'w'), indent=2)

cm = confusion_matrix(y_true, y_pred)
cmn = cm / cm.sum(1, keepdims=True)
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, mat, tit, fmt in [(axs[0], cm, 'Contagens', '{:d}'), (axs[1], cmn, 'Normalizada por linha', '{:.2f}')]:
    ax.imshow(mat, cmap='Blues'); ax.set_title(tit)
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(nm, rotation=20); ax.set_yticklabels(nm)
    ax.set_xlabel('previsto'); ax.set_ylabel('real')
    for a in range(3):
        for b in range(3):
            ax.text(b, a, fmt.format(mat[a, b]), ha='center', va='center',
                    color='white' if mat[a, b] > mat.max() / 2 else 'black')
plt.tight_layout()
os.makedirs(f'{BASE}/results/confusion_matrix', exist_ok=True)
plt.savefig(f'{BASE}/results/confusion_matrix/matriz_confusao_teste.png', dpi=150); plt.show()

pred = pd.DataFrame(dict(id=ids, real=[nomes[k] for k in y_true], previsto=[nomes[k] for k in y_pred],
                         prob_prevista=probs.max(1), prob_meningioma=probs[:, 0],
                         prob_glioma=probs[:, 1], prob_pituitary=probs[:, 2],
                         acertou=(y_true == y_pred)))
pred.to_csv(f'{MET_DIR}/predicoes_teste.csv', index=False)
print('Erros:', int((~pred.acertou).sum()), 'de', len(pred))